In [8]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path when running from example_notebooks
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import pandas as pd
import optuna
import pickle
from functools import partial
from pathlib import Path

from simulator.simulation.modules import Campaign
from simulator.simulation.utils_visualization import data_prep_vis, plot_history_article
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import autobidder_check

In [10]:
from simulator.model.rlb_dp_bidder import RLBDPBidder

In [11]:
auction_mode = "FPA"  # or "VCG"
best_params_subfolder = f"{auction_mode.lower()}_rlb_n10_rndm_42"
best_models_subfolder = f"{auction_mode.lower()}_rlb_n10_rndm_42"

# metric to optimize: CPC_REL / RMSE / SCR
metric = "SCR"
n_trials = 10

In [12]:
data_dir = project_root / "data" / auction_mode.lower()

data_config = {
    "train": {
        "campaigns_path": str(data_dir / f"campaigns_{auction_mode.lower()}_filtered_train_final.csv"),
        "stats_path": str(data_dir / f"stats_{auction_mode.lower()}_filtered_train_final.csv"),
    },
    "test": {
        "campaigns_path": str(data_dir / f"campaigns_{auction_mode.lower()}_filtered_test_final.csv"),
        "stats_path": str(data_dir / f"stats_{auction_mode.lower()}_filtered_test_final.csv"),
    },
}

data_config

{'train': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_filtered_train_final.csv',
  'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_filtered_train_final.csv'},
 'test': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_filtered_test_final.csv',
  'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_filtered_test_final.csv'}}

In [15]:
Path.cwd()

PosixPath('/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks')

In [16]:
project_root

PosixPath('/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark')

In [ ]:
import sys
from pathlib import Path

# Robust paths even if you run this cell first in a fresh kernel
if "project_root" not in globals():
    project_root = Path.cwd().resolve().parent
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

sys.path.append(str(project_root / "example_notebooks" / "evaluate_baselines"))

from baselines_finetune import BaseLineTrainer

baseline_params_subfolder = "fpa_baseline_n10_rndm_42"

In [22]:
baseline_trainer = BaseLineTrainer(
    data_config=data_config,
    metric=metric,
    auction_mode=auction_mode,
    base_params_subfolder=baseline_params_subfolder,
    random_state=42,
)

# baselines_finetune builds best_params path relative to CWD; pin it to evaluate_baselines dir
baseline_params_dir = (
    project_root / "example_notebooks" / "evaluate_baselines" / "best_params" / baseline_params_subfolder
)
baseline_trainer._get_params_path = lambda model_name: str(
    baseline_params_dir / f"{model_name}_{metric.lower()}_{auction_mode}.pkl"
)

In [37]:
# models = ["linear", "tapid", "mpid", "broi"]
models = ["linear", "tapid", "broi"]

baseline_results = []
for model_name in models:
    objective_fn = getattr(baseline_trainer, f"objective_{model_name}")
    value = objective_fn(None, eval=True)

    cpc_rel, rmse, scr, _ = value
    metric_value = {"CPC_REL": cpc_rel, "RMSE": rmse, "SCR": scr}[metric]


    baseline_results.append(
        {
            "model": model_name,
            "metric": metric,
            "objective_value": metric_value,
            "scores": value,
            "cpc_rel": cpc_rel,
            "rmse": rmse,
            "scr": scr,
            "params_subfolder": baseline_params_subfolder,
        }
    )

# baseline_results_df = pd.DataFrame(baseline_results)
# baseline_results_df.sort_values("objective_value", ascending=(metric != "SCR"))

CPC_REL: 182.69371981120636, rmse: 1.3087288172293368, SCR: 38000.766231548245
CPC_REL: 11.962526643816842, rmse: 3.004924969587723, SCR: 33976.18643112706
CPC_REL: 1673.199264722307, rmse: 1.2203193763993325, SCR: 16290.10399082903


In [38]:
baseline_results

[{'model': 'linear',
  'metric': 'SCR',
  'objective_value': 38000.766231548245,
  'scores': (182.69371981120636,
   1.3087288172293368,
   38000.766231548245,
   0.0007782101167315176),
  'cpc_rel': 182.69371981120636,
  'rmse': 1.3087288172293368,
  'scr': 38000.766231548245,
  'params_subfolder': 'fpa_baseline_n10_rndm_42'},
 {'model': 'tapid',
  'metric': 'SCR',
  'objective_value': 33976.18643112706,
  'scores': (11.962526643816842,
   3.004924969587723,
   33976.18643112706,
   0.007003891050583658),
  'cpc_rel': 11.962526643816842,
  'rmse': 3.004924969587723,
  'scr': 33976.18643112706,
  'params_subfolder': 'fpa_baseline_n10_rndm_42'},
 {'model': 'broi',
  'metric': 'SCR',
  'objective_value': 16290.10399082903,
  'scores': (1673.199264722307, 1.2203193763993325, 16290.10399082903, 0.0),
  'cpc_rel': 1673.199264722307,
  'rmse': 1.2203193763993325,
  'scr': 16290.10399082903,
  'params_subfolder': 'fpa_baseline_n10_rndm_42'}]

In [39]:
baseline_results_df = pd.DataFrame(baseline_results)

In [42]:
baseline_results_df['scr'] = baseline_results_df['scr'].apply(lambda x: round(x,2))

In [43]:
baseline_results_df[['model','metric','cpc_rel','rmse','scr','params_subfolder']]

,model,metric,cpc_rel,rmse,scr,params_subfolder
0,linear,SCR,182.693720,1.308729,38000.77,fpa_baseline_n10_rndm_42
1,tapid,SCR,11.962527,3.004925,33976.19,fpa_baseline_n10_rndm_42
2,broi,SCR,1673.199265,1.220319,16290.10,fpa_baseline_n10_rndm_42


In [58]:
baseline_results_df[['model','metric','cpc_rel','rmse','scr','params_subfolder']].to_json(orient='records')

'[{"model":"linear","metric":"SCR","cpc_rel":182.6937198112,"rmse":1.3087288172,"scr":38000.77,"params_subfolder":"fpa_baseline_n10_rndm_42"},{"model":"tapid","metric":"SCR","cpc_rel":11.9625266438,"rmse":3.0049249696,"scr":33976.19,"params_subfolder":"fpa_baseline_n10_rndm_42"},{"model":"broi","metric":"SCR","cpc_rel":1673.1992647223,"rmse":1.2203193764,"scr":16290.1,"params_subfolder":"fpa_baseline_n10_rndm_42"}]'

CPC_REL: 328.38205798055765, rmse: 1.2777773701938937, SCR: 32566.61670594752

> ON TEST DATA RLB

In [73]:
stats_path_test = data_config['test']['stats_path']
campaigns_path_test = data_config['test']['campaigns_path']

In [74]:
stats_test_df = pd.read_csv(stats_path_test)
campaigns_test_df = pd.read_csv(campaigns_path_test)

In [75]:
!pwd

/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks


In [76]:
rlb_best_params_finetune = 'tune_rlb/best_params/rlb_dp_scr_FPA.pkl'

In [80]:
best_model_path = f'tune_rlb/best_models/{best_models_subfolder}/{metric.lower()}.pkl'

In [81]:
best_params_rlb = pd.read_pickle(rlb_best_params_finetune)
best_model_rlb_path = best_model_path

In [82]:
res = autobidder_check(
    bidder=RLBDPBidder,
    params = {
        "input_campaigns": campaigns_path_test,
        "input_stats": stats_path_test,
        "model_path": best_model_path,
        **best_params_rlb
    },
    auction_mode=auction_mode,
)

(cpc_relative, rmse, clicks_sum, quickspend)

In [86]:
cpc_rel, rmse, scr, _ = res['score']

In [87]:
baseline_results_df = pd.concat(
    [
        baseline_results_df, 
        pd.DataFrame([
            {
                'model': 'rlb',
                'metric': metric,
                'objective_value': scr if metric == 'SCR' else (cpc_rel if metric == 'CPC_REL' else rmse),
                'scores': res['score'],
                'cpc_rel': cpc_rel,
                'rmse': rmse,
                'scr': scr,
                'params_subfolder': baseline_params_subfolder
            }
        ])
    ], ignore_index=True
)

In [88]:
baseline_results_df

,model,metric,objective_value,scores,cpc_rel,rmse,scr,params_subfolder
0,linear,SCR,38000.766232,"(182.69371981120636, 1.3087288172293368, 38000...",182.693720,1.308729,38000.770000,fpa_baseline_n10_rndm_42
1,tapid,SCR,33976.186431,"(11.962526643816842, 3.004924969587723, 33976....",11.962527,3.004925,33976.190000,fpa_baseline_n10_rndm_42
2,broi,SCR,16290.103991,"(1673.199264722307, 1.2203193763993325, 16290....",1673.199265,1.220319,16290.100000,fpa_baseline_n10_rndm_42
3,rlb,SCR,32396.078021,"(328.5357522422687, 1.3405143675549405, 32396....",328.535752,1.340514,32396.078021,fpa_baseline_n10_rndm_42


In [89]:
baseline_results_df[['model','metric','cpc_rel','rmse','scr','params_subfolder']]

,model,metric,cpc_rel,rmse,scr,params_subfolder
0,linear,SCR,182.693720,1.308729,38000.770000,fpa_baseline_n10_rndm_42
1,tapid,SCR,11.962527,3.004925,33976.190000,fpa_baseline_n10_rndm_42
2,broi,SCR,1673.199265,1.220319,16290.100000,fpa_baseline_n10_rndm_42
3,rlb,SCR,328.535752,1.340514,32396.078021,fpa_baseline_n10_rndm_42
